# Kuroshio Daily R2 FIX2 — TCN Flatten/FC 维度修复 + 训练前结构预检 + 断点续训
## RTX 5080 / WSL / VS Code

修复 `RuntimeError: mat1 and mat2 shapes cannot be multiplied (6x960 and 60x32)`。

根因：Branch 1 的 TCN-like 卷积输出为 16 个通道 × 60 个位置，flatten 后是 960；旧代码的全连接层却只按 60 构造。FIX2 改成 `flat_dim = out_channels * conv_output_length`，并增加全模型 preflight、断点续训与 partial metrics。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, json, math, random, warnings, copy
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"
    DATASET_FILE: str = "dataset/daily_dual_branch_paths.npz"

    # 首选仍为180天，保持“约6个月历史”的原任务语义。
    # 如果R1完成路径存在缺日，FIX1只通过缩短历史窗做资格验证，
    # 不会对forecast target做插值。
    LOOKBACK_DAYS: int = 180
    LOOKBACK_CANDIDATES: tuple = (180, 120, 90, 60, 30)
    AUTO_LOOKBACK_FALLBACK: bool = True

    LEAD_DAYS: tuple = (30, 60, 90)

    # 资格验证时间块；最终论文需扩展更多年份后做 rolling hindcast
    TRAIN_TARGET_END: str = "2024-08-31"
    VAL_TARGET_END: str = "2024-11-30"

    # 原代码第一分支实际使用 31 个点（43点中取13:43）
    LEGACY_BRANCH1_31PTS: bool = True

    # 原代码第二分支只取前229点，即大约122E~141E。
    # 为与论文“延伸至145E”更一致，默认 False = 使用122E~145E全域。
    LEGACY_BRANCH2_229PTS: bool = False

    EOF_MODES_B1: int = 31
    EOF_MODES_B2: int = 30

    # 输入历史窗口允许极短缺口；目标日始终必须真实存在。
    MAX_INPUT_GAP_DAYS: int = 2
    MAX_INPUT_MISSING_FRAC: float = 0.03

    # qualification最低anchor数。达不到就不进入正式神经网络训练。
    MIN_TRAIN_ANCHORS: int = 24
    MIN_VAL_ANCHORS: int = 8
    MIN_TEST_ANCHORS: int = 8

    BATCH_SIZE: int = 32
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    DROPOUT: float = 0.30
    EPOCHS_B1: int = 150
    EPOCHS_B2: int = 100
    LR_DROP_B1: int = 20
    LR_DROP_B2: int = 65
    EARLY_STOP_PATIENCE: int = 18

    SEEDS: tuple = (20, 21, 22)
    AR_LAGS: int = 7

    RESUME_COMPLETED_RUNS: bool = True
    ALLOW_TRUSTED_LEGACY_RESUME: bool = True
    PREFLIGHT_BATCH: int = 2

cfg = CFG()
ROOT = Path(cfg.PROJECT_ROOT)
DATAFILE = ROOT / cfg.DATASET_FILE
OUT = ROOT / "R2_prediction"
for d in [OUT, OUT/"models", OUT/"reports", OUT/"figures"]:
    d.mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False, default=list))
print("DATAFILE:", DATAFILE)


In [ ]:

# ===== 1. 环境与GPU检查 =====
import sys, platform, subprocess
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
    print("Capability:", torch.cuda.get_device_capability(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:

# ===== 2. 读取R1双分支逐日路径 + 基础覆盖审计 =====
if not DATAFILE.exists():
    raise FileNotFoundError(
        f"未找到 R1 数据集: {DATAFILE}\n"
        "请先完成 R1/FIX3 的双分支路径导出。"
    )

z = np.load(DATAFILE)
dates = pd.to_datetime(z["dates"])
B1 = z["branch1_lon"].astype(np.float32)
LAT1 = z["branch1_target_lats"].astype(np.float64)
B2 = z["branch2_lat"].astype(np.float32)
LON2 = z["branch2_target_lons"].astype(np.float64)

order = np.argsort(dates)
dates, B1, B2 = dates[order], B1[order], B2[order]

# 去掉重复日期，保留最后一条
dups = pd.Index(dates).duplicated(keep="last")
if dups.any():
    print(f"[WARN] 发现重复日期 {int(dups.sum())} 条，保留每个日期最后一条。")
    keep = ~dups
    dates, B1, B2 = dates[keep], B1[keep], B2[keep]

if cfg.LEGACY_BRANCH1_31PTS:
    B1 = B1[:, 12:]
    LAT1 = LAT1[12:]

if cfg.LEGACY_BRANCH2_229PTS:
    B2 = B2[:, :229]
    LON2 = LON2[:229]

if len(dates) == 0:
    raise RuntimeError("daily_dual_branch_paths.npz 中没有任何日期。")

calendar_days = int((dates.max() - dates.min()).days + 1)
coverage = len(dates) / max(calendar_days, 1)

print("date:", dates.min(), "->", dates.max(), "N=", len(dates))
print("calendar days:", calendar_days)
print("observed path coverage:", f"{coverage:.3%}")
print("Branch1:", B1.shape, "lat:", LAT1[0], "->", LAT1[-1])
print("Branch2:", B2.shape, "lon:", LON2[0], "->", LON2[-1])

# 每条路径内部NaN审计
b1_bad = np.isnan(B1).any(axis=1)
b2_bad = np.isnan(B2).any(axis=1)
print("B1 rows containing NaN:", int(b1_bad.sum()))
print("B2 rows containing NaN:", int(b2_bad.sum()))

coverage_report = pd.DataFrame({
    "date": dates,
    "b1_has_nan": b1_bad,
    "b2_has_nan": b2_bad,
})
coverage_report.to_csv(
    OUT/"reports"/"R2_daily_path_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:

# ===== 3. Anchor构建：先审计180天，再按最长可行lookback自动回退 =====
full_dates = pd.date_range(dates.min(), dates.max(), freq="D")

def matrix_to_daily_df(mat, cols):
    df = pd.DataFrame(
        mat,
        index=pd.DatetimeIndex(dates).normalize(),
        columns=cols
    )
    return df.reindex(full_dates)

df1 = matrix_to_daily_df(
    B1, [f"b1_{i}" for i in range(B1.shape[1])]
)
df2 = matrix_to_daily_df(
    B2, [f"b2_{i}" for i in range(B2.shape[1])]
)

present = pd.Series(False, index=full_dates)
present.loc[pd.DatetimeIndex(dates).normalize()] = True

def longest_missing_run(mask_missing):
    best = cur = 0
    for v in mask_missing:
        if bool(v):
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best

# 整体缺日诊断
missing_calendar = ~present.to_numpy()
print("calendar missing days:", int(missing_calendar.sum()))
print("longest calendar missing run:", longest_missing_run(missing_calendar))
print("calendar missing fraction:", f"{missing_calendar.mean():.3%}")

def get_input_window(df, issue_date, lookback_days, reason_counter=None):
    start = issue_date - pd.Timedelta(days=lookback_days-1)
    win = df.loc[start:issue_date].copy()

    if len(win) != lookback_days:
        if reason_counter is not None:
            reason_counter["short_calendar_window"] += 1
        return None

    row_missing = win.isna().any(axis=1).to_numpy()
    miss_frac = float(row_missing.mean())
    max_gap = longest_missing_run(row_missing)

    if miss_frac > cfg.MAX_INPUT_MISSING_FRAC:
        if reason_counter is not None:
            reason_counter["input_missing_fraction"] += 1
        return None

    if max_gap > cfg.MAX_INPUT_GAP_DAYS:
        if reason_counter is not None:
            reason_counter["input_long_gap"] += 1
        return None

    # 只在历史输入窗口内部补极短缺口。
    # 这些点都发生在issue之前，因此不会读取未来forecast target。
    win = win.interpolate(
        method="time",
        limit=cfg.MAX_INPUT_GAP_DAYS,
        limit_area="inside"
    )
    win = (
        win.ffill(limit=cfg.MAX_INPUT_GAP_DAYS)
           .bfill(limit=cfg.MAX_INPUT_GAP_DAYS)
    )

    if win.isna().any().any():
        if reason_counter is not None:
            reason_counter["input_unfilled_nan"] += 1
        return None

    return win.to_numpy(dtype=np.float32)

def actual_target(df, target_date):
    # target绝不插值
    if target_date not in df.index:
        return None
    if not bool(present.loc[target_date]):
        return None

    row = df.loc[target_date]
    if row.isna().any():
        return None

    return row.to_numpy(dtype=np.float32)

def classify_split(tdates):
    train_end = pd.Timestamp(cfg.TRAIN_TARGET_END)
    val_end = pd.Timestamp(cfg.VAL_TARGET_END)

    tmin, tmax = min(tdates), max(tdates)

    if tmax <= train_end:
        return "train"
    elif tmin > train_end and tmax <= val_end:
        return "val"
    elif tmin > val_end:
        return "test"
    return "gap"

def build_anchors_for_lookback(lookback_days):
    max_lead = max(cfg.LEAD_DAYS)
    min_issue = full_dates.min() + pd.Timedelta(days=lookback_days-1)
    max_issue = full_dates.max() - pd.Timedelta(days=max_lead)

    reasons = {
        "candidate_issue_days": 0,
        "input_missing_b1": 0,
        "input_missing_b2": 0,
        "target_missing_b1": 0,
        "target_missing_b2": 0,
        "short_calendar_window": 0,
        "input_missing_fraction": 0,
        "input_long_gap": 0,
        "input_unfilled_nan": 0,
    }

    anchors_local = []

    if min_issue > max_issue:
        return anchors_local, reasons

    for issue in pd.date_range(min_issue, max_issue, freq="D"):
        reasons["candidate_issue_days"] += 1

        local_reasons_1 = {
            "short_calendar_window":0,
            "input_missing_fraction":0,
            "input_long_gap":0,
            "input_unfilled_nan":0,
        }
        local_reasons_2 = dict(local_reasons_1)

        x1 = get_input_window(
            df1, issue, lookback_days, local_reasons_1
        )
        x2 = get_input_window(
            df2, issue, lookback_days, local_reasons_2
        )

        for k,v in local_reasons_1.items():
            reasons[k] += v
        for k,v in local_reasons_2.items():
            reasons[k] += v

        if x1 is None:
            reasons["input_missing_b1"] += 1
            continue
        if x2 is None:
            reasons["input_missing_b2"] += 1
            continue

        tdates = [
            issue + pd.Timedelta(days=d)
            for d in cfg.LEAD_DAYS
        ]

        y1 = [actual_target(df1,d) for d in tdates]
        y2 = [actual_target(df2,d) for d in tdates]

        if any(v is None for v in y1):
            reasons["target_missing_b1"] += 1
            continue
        if any(v is None for v in y2):
            reasons["target_missing_b2"] += 1
            continue

        anchors_local.append(
            (
                issue,
                tdates,
                x1,
                x2,
                np.stack(y1),
                np.stack(y2),
            )
        )

    return anchors_local, reasons

def split_counts_for(anchors_local):
    out = {"train":0, "val":0, "test":0, "gap":0}
    for a in anchors_local:
        out[classify_split(a[1])] += 1
    return out

candidate_rows = []
candidate_store = {}

lookbacks = (
    cfg.LOOKBACK_CANDIDATES
    if cfg.AUTO_LOOKBACK_FALLBACK
    else (cfg.LOOKBACK_DAYS,)
)

for lb in lookbacks:
    aa, rr = build_anchors_for_lookback(int(lb))
    ss = split_counts_for(aa)
    candidate_store[int(lb)] = aa

    candidate_rows.append({
        "lookback_days": int(lb),
        "anchors_total": len(aa),
        "train": ss["train"],
        "val": ss["val"],
        "test": ss["test"],
        "gap": ss["gap"],
        **rr,
    })

anchor_audit = pd.DataFrame(candidate_rows)
anchor_audit.to_csv(
    OUT/"reports"/"anchor_lookback_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n===== Anchor / Lookback 资格审计 =====")
display(anchor_audit)

def qualifies(row):
    return (
        row["train"] >= cfg.MIN_TRAIN_ANCHORS and
        row["val"] >= cfg.MIN_VAL_ANCHORS and
        row["test"] >= cfg.MIN_TEST_ANCHORS
    )

qualified = anchor_audit[
    anchor_audit.apply(qualifies, axis=1)
].copy()

if len(qualified):
    # LOOKBACK_CANDIDATES按从长到短排列，选择最长合格历史窗
    selected_lookback = int(qualified.iloc[0]["lookback_days"])
else:
    # 不强行训练。先输出最佳候选，帮助定位究竟差多少。
    best = anchor_audit.sort_values(
        ["anchors_total","train","val","test"],
        ascending=False
    ).iloc[0]

    raise RuntimeError(
        "R2 FIX1 Anchor Gate失败：当前R1逐日路径不足以构造可靠的"
        " train/val/test 时序样本。\n"
        f"最佳候选 lookback={int(best['lookback_days'])} 天，"
        f"anchors={int(best['anchors_total'])}, "
        f"train={int(best['train'])}, "
        f"val={int(best['val'])}, "
        f"test={int(best['test'])}。\n"
        "请查看：R2_prediction/reports/anchor_lookback_audit.csv。\n"
        "不要通过随机切分日期或插值forecast target来绕过这个Gate。"
    )

cfg.LOOKBACK_DAYS = selected_lookback
anchors = candidate_store[selected_lookback]

print("\n===== R2 FIX1最终选择 =====")
print("selected LOOKBACK_DAYS =", cfg.LOOKBACK_DAYS)
print("anchors =", len(anchors))
print("split counts =", split_counts_for(anchors))

with open(
    OUT/"reports"/"selected_temporal_protocol.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "selected_lookback_days": int(cfg.LOOKBACK_DAYS),
            "lead_days": list(cfg.LEAD_DAYS),
            "max_input_gap_days": int(cfg.MAX_INPUT_GAP_DAYS),
            "max_input_missing_frac": float(cfg.MAX_INPUT_MISSING_FRAC),
            "train_target_end": cfg.TRAIN_TARGET_END,
            "val_target_end": cfg.VAL_TARGET_END,
            "split_counts": split_counts_for(anchors),
        },
        f,
        indent=2,
        ensure_ascii=False,
    )


In [ ]:

# ===== 4. 严格按目标时间块切 train / val / test =====
train_end = pd.Timestamp(cfg.TRAIN_TARGET_END)
val_end = pd.Timestamp(cfg.VAL_TARGET_END)

split_rows = []

for j,a in enumerate(anchors):
    issue,tdates,*_ = a
    tmin,tmax = min(tdates),max(tdates)

    split = classify_split(tdates)

    split_rows.append({
        "idx":j,
        "issue":issue,
        "target_min":tmin,
        "target_max":tmax,
        "split":split,
    })

# 即使为空也固定columns，避免再出现KeyError。
# 但正常情况下上一单元的Anchor Gate已经确保这里非空。
split_df = pd.DataFrame(
    split_rows,
    columns=[
        "idx","issue","target_min","target_max","split"
    ]
)

if split_df.empty:
    raise RuntimeError(
        "split_df为空。上一单元本应被Anchor Gate截停；"
        "请从第3节重新顺序运行Notebook。"
    )

split_summary = (
    split_df.groupby("split", dropna=False)
    .agg(
        n=("idx","size"),
        issue_min=("issue","min"),
        issue_max=("issue","max"),
        target_min=("target_min","min"),
        target_max=("target_max","max"),
    )
)

display(split_summary)

# 强制检查三块都有样本
required = {"train","val","test"}
available = set(split_df.loc[
    split_df["split"].isin(required), "split"
].unique())

missing_splits = required - available
if missing_splits:
    raise RuntimeError(
        f"时间块缺失: {sorted(missing_splits)}。"
        "请查看 anchor_lookback_audit.csv，不要继续训练。"
    )

split_df.to_csv(
    OUT/"reports"/"time_split.csv",
    index=False,
    encoding="utf-8-sig"
)

print("time split saved ->", OUT/"reports"/"time_split.csv")



## 5. Train-only scaler / EOF

旧 MATLAB 源码中均值、标准差和 EOF 均由训练集计算；本 Notebook 保留这一点。

FIX1 的 `selected_lookback_days` 只决定历史输入长度，不改变：

```text
+30 d / +60 d / +90 d
```

三个预测目标。

验证和测试日期仍不参与 scaler/EOF 拟合。最终选择的 lookback 会写入：

```text
R2_prediction/reports/selected_temporal_protocol.json
```

后续写论文时必须按照这个实际值描述，不能仍然笼统写“180 days”。


In [ ]:

# ===== 5. Train-only scaler + EOF =====
def fit_preprocess(df, train_end, n_modes):
    train = df.loc[:train_end].dropna()
    arr = train.to_numpy(dtype=np.float64)
    mu = arr.mean(axis=0)
    sd = arr.std(axis=0, ddof=1)
    sd[sd < 1e-8] = 1.0
    X = (arr-mu)/sd
    # [time,space] = U S Vt；EOF空间模态为V
    U,S,Vt=np.linalg.svd(X, full_matrices=False)
    modes=min(n_modes,Vt.shape[0],Vt.shape[1])
    V=Vt[:modes].T
    exp=(S**2)/(np.sum(S**2)+1e-12)
    return {"mu":mu.astype(np.float32),"sd":sd.astype(np.float32),
            "V":V.astype(np.float32),"explained":exp[:modes].astype(np.float32)}

pp1=fit_preprocess(df1,train_end,cfg.EOF_MODES_B1)
pp2=fit_preprocess(df2,train_end,cfg.EOF_MODES_B2)
print("B1 EOF modes:",pp1["V"].shape[1],"cum=",pp1["explained"].sum())
print("B2 EOF modes:",pp2["V"].shape[1],"cum=",pp2["explained"].sum())

def standardize(x,pp):
    return (x-pp["mu"])/(pp["sd"]+1e-8)
def to_pc(x,pp):
    return standardize(x,pp) @ pp["V"]
def from_pc(pc,pp):
    return (pc @ pp["V"].T)*pp["sd"] + pp["mu"]


In [ ]:

# ===== 6. 转为训练张量：原序列 + 日一阶差分 =====
def make_arrays(branch, pp):
    rows=[]
    for r in split_df.itertuples(index=False):
        if r.split not in ("train","val","test"):
            continue
        a=anchors[r.idx]
        issue,tdates,x1,x2,y1,y2=a
        x = x1 if branch==1 else x2
        y = y1 if branch==1 else y2
        xs=standardize(x,pp).astype(np.float32)
        xd=np.diff(xs,axis=0,prepend=xs[:1])
        xin=np.concatenate([xs,xd],axis=1)  # [T,2M]
        ypc=to_pc(y,pp).astype(np.float32)   # [3,modes]
        rows.append((r.split, issue, xin, ypc, y.astype(np.float32)))
    out={}
    for s in ("train","val","test"):
        q=[x for x in rows if x[0]==s]
        if not q:
            out[s]=None; continue
        out[s]={
            "issue":np.array([x[1] for x in q],dtype="datetime64[D]"),
            "X":np.stack([x[2] for x in q]),
            "Ypc":np.stack([x[3] for x in q]),
            "Ypath":np.stack([x[4] for x in q]),
        }
    return out

D1=make_arrays(1,pp1)
D2=make_arrays(2,pp2)
for s in ("train","val","test"):
    print(s, None if D1[s] is None else D1[s]["X"].shape,
          None if D2[s] is None else D2[s]["X"].shape)


In [ ]:
# ===== 7. PyTorch模型 — FIX2：修复TCN flatten→FC维度 =====
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class PerStepConvEncoder(nn.Module):
    def __init__(self, feature_dim, variant, branch, dropout=0.3):
        super().__init__()
        self.variant = variant
        self.branch = branch
        self.feature_dim = int(feature_dim)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)

        if variant == "conv":
            c1 = 16 if branch == 1 else 32
            c2 = 32 if branch == 1 else 64
            self.conv1 = nn.Conv1d(1, c1, 3, padding=1)
            self.conv2 = nn.Conv1d(c1, c2, 3, padding=1)
            self.bn1 = None
            self.fc = None
            self.encoder_out_dim = c2 * self.feature_dim

        elif variant == "tcn":
            c = 16 if branch == 1 else 32
            k = 3 if branch == 1 else 5
            hidden = 32 if branch == 1 else 64
            self.k = k
            self.out_channels = c
            self.conv1 = nn.Conv1d(1, c, kernel_size=k, dilation=2, padding=0)
            self.bn1 = nn.BatchNorm1d(c)
            self.conv2 = None
            conv_len = self._conv_out_len(self.feature_dim, k)
            if conv_len <= 0:
                raise ValueError(f"TCN conv output length <= 0: F={feature_dim}, k={k}")
            flat_dim = c * conv_len
            self.fc = nn.Linear(flat_dim, hidden)
            self.encoder_out_dim = hidden
        else:
            raise ValueError(variant)

    @staticmethod
    def _conv_out_len(L, k):
        effective_kernel = (k - 1) * 2 + 1
        return (L + 2) - effective_kernel + 1

    def forward(self, x):
        B,T,F = x.shape
        if F != self.feature_dim:
            raise RuntimeError(f"expected feature_dim={self.feature_dim}, got {F}")
        z = x.reshape(B*T, 1, F)
        if self.variant == "conv":
            z = self.drop(self.relu(self.conv1(z)))
            z = self.drop(self.relu(self.conv2(z)))
            z = z.flatten(1)
        else:
            z = torch.nn.functional.pad(z, (2,0))
            z = self.conv1(z)
            z = self.drop(self.relu(self.bn1(z)))
            z = z.flatten(1)
            if z.shape[1] != self.fc.in_features:
                raise RuntimeError(
                    f"TCN flatten mismatch: actual={z.shape[1]}, expected={self.fc.in_features}, "
                    f"F={self.feature_dim}, channels={self.out_channels}, k={self.k}"
                )
            z = self.drop(self.relu(self.fc(z)))
        return z.reshape(B,T,-1)

class HybridNet(nn.Module):
    def __init__(self, feature_dim, n_modes, branch, conv_kind, rnn_kind, dropout=0.3):
        super().__init__()
        self.enc = PerStepConvEncoder(feature_dim, conv_kind, branch, dropout)
        edim = self.enc.encoder_out_dim
        hidden = (16 if branch == 1 else 64) if conv_kind == "conv" else (32 if branch == 1 else 64)
        RNN = nn.LSTM if rnn_kind == "lstm" else nn.GRU
        self.rnn = RNN(edim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, len(cfg.LEAD_DAYS)*n_modes)
        self.n_modes = n_modes

    def forward(self, x):
        z = self.enc(x)
        o,_ = self.rnn(z)
        return self.head(o[:,-1]).reshape(x.shape[0], len(cfg.LEAD_DAYS), self.n_modes)

class SimpleLSTM(nn.Module):
    def __init__(self, feature_dim, n_modes, branch):
        super().__init__()
        hidden = 32 if branch == 1 else 64
        self.rnn = nn.LSTM(feature_dim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, len(cfg.LEAD_DAYS)*n_modes)
        self.n_modes = n_modes

    def forward(self, x):
        o,_ = self.rnn(x)
        return self.head(o[:,-1]).reshape(x.shape[0], len(cfg.LEAD_DAYS), self.n_modes)

def build_model(name, conv_kind, rnn_kind, branch):
    data = D1 if branch == 1 else D2
    pp = pp1 if branch == 1 else pp2
    feature_dim = int(data["train"]["X"].shape[-1])
    n_modes = int(pp["V"].shape[1])
    if conv_kind == "simple":
        return SimpleLSTM(feature_dim, n_modes, branch)
    return HybridNet(feature_dim, n_modes, branch, conv_kind, rnn_kind, cfg.DROPOUT)

PREFLIGHT_SPECS = [
    ("Simple-LSTM","simple","lstm"),
    ("EOF-Conv-LSTM","conv","lstm"),
    ("EOF-Conv-GRU","conv","gru"),
    ("EOF-TCN-LSTM","tcn","lstm"),
    ("EOF-TCN-GRU","tcn","gru"),
]
preflight_rows = []
for name,conv_kind,rnn_kind in PREFLIGHT_SPECS:
    for branch,data,pp in [(1,D1,pp1),(2,D2,pp2)]:
        model = build_model(name,conv_kind,rnn_kind,branch)
        nsmoke = min(cfg.PREFLIGHT_BATCH, len(data["train"]["X"]))
        xb = torch.from_numpy(data["train"]["X"][:nsmoke]).float()
        with torch.no_grad():
            yb = model(xb)
        expected = (len(xb), len(cfg.LEAD_DAYS), pp["V"].shape[1])
        if tuple(yb.shape) != tuple(expected):
            raise RuntimeError(f"PRE-FLIGHT FAIL: {name}, B{branch}, got={tuple(yb.shape)}, expected={expected}")
        row = {
            "model": name,
            "branch": branch,
            "input_shape": str(tuple(xb.shape)),
            "output_shape": str(tuple(yb.shape)),
            "params": sum(p.numel() for p in model.parameters()),
            "status": "PASS",
        }
        if hasattr(model, "enc") and model.enc.variant == "tcn":
            row["tcn_fc_in"] = model.enc.fc.in_features
        preflight_rows.append(row)
preflight_df = pd.DataFrame(preflight_rows)
display(preflight_df)
preflight_df.to_csv(OUT/"reports"/"architecture_preflight_FIX2.csv", index=False)
print("[R2 FIX2 architecture preflight] ALL PASS")


In [ ]:

# ===== 8. 训练函数：validation + early stopping + seed stability =====
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def dl(arr,shuffle=False):
    ds=TensorDataset(torch.from_numpy(arr["X"]).float(),
                     torch.from_numpy(arr["Ypc"]).float())
    return DataLoader(ds,batch_size=cfg.BATCH_SIZE,shuffle=shuffle,
                      num_workers=0,pin_memory=torch.cuda.is_available())

def train_one(model, data, epochs, lr_drop, seed):
    set_seed(seed)
    model=model.to(device)
    opt=torch.optim.Adam(model.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.StepLR(opt,step_size=lr_drop,gamma=0.5)
    crit=nn.MSELoss()
    tr_loader=dl(data["train"],True)
    va_loader=dl(data["val"],False)
    best={"loss":float("inf"),"state":None,"epoch":0}
    bad=0; hist=[]
    for ep in range(1,epochs+1):
        model.train(); sm=0.; n=0
        for xb,yb in tr_loader:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(set_to_none=True)
            pred=model(xb); loss=crit(pred,yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step()
            sm += loss.detach().item()*len(xb); n += len(xb)
        tr=sm/max(n,1)
        model.eval(); sm=0.; n=0
        with torch.no_grad():
            for xb,yb in va_loader:
                xb,yb=xb.to(device),yb.to(device)
                loss=crit(model(xb),yb)
                sm += loss.detach().item()*len(xb); n += len(xb)
        va=sm/max(n,1)
        hist.append((ep,tr,va,opt.param_groups[0]["lr"]))
        if va < best["loss"]-1e-7:
            best={"loss":va,"state":copy.deepcopy(model.state_dict()),"epoch":ep}; bad=0
        else:
            bad+=1
        sch.step()
        if ep==1 or ep%10==0:
            print(f"seed={seed} ep={ep:03d} train={tr:.6f} val={va:.6f}")
        if bad>=cfg.EARLY_STOP_PATIENCE:
            break
    model.load_state_dict(best["state"])
    return model, pd.DataFrame(hist,columns=["epoch","train_loss","val_loss","lr"]), best

@torch.no_grad()
def predict_pc(model, arr):
    model.eval()
    loader=DataLoader(torch.from_numpy(arr["X"]).float(),
                      batch_size=cfg.BATCH_SIZE,shuffle=False)
    out=[]
    for xb in loader:
        out.append(model(xb.to(device)).cpu().numpy())
    return np.concatenate(out)


In [ ]:

# ===== 9. 路径拼接（禁止使用真实值帮助预测拼接）+ cross-track km =====
EARTH_R=6371.0088

def join_path(b1_lon,b2_lat):
    # branch1: fixed LAT1; branch2: fixed LON2
    end_lon=float(b1_lon[-1]); end_lat=float(LAT1[-1])
    valid=np.where(LON2 > end_lon)[0]
    if len(valid)==0:
        valid=np.arange(len(LON2))
    # 仅使用预测本身的几何关系，不读取truth
    lat_scale=111.32
    lon_scale=111.32*np.cos(np.deg2rad(end_lat))
    dist=np.hypot((LON2[valid]-end_lon)*lon_scale,(b2_lat[valid]-end_lat)*lat_scale)
    st=valid[int(np.argmin(dist))]
    lon=np.concatenate([b1_lon,LON2[st:]])
    lat=np.concatenate([LAT1,b2_lat[st:]])
    return lon,lat

def to_xy_km(lon,lat,lat0=None,lon0=None):
    lon=np.asarray(lon); lat=np.asarray(lat)
    if lat0 is None: lat0=np.nanmean(lat)
    if lon0 is None: lon0=np.nanmean(lon)
    x=(lon-lon0)*111.32*np.cos(np.deg2rad(lat0))
    y=(lat-lat0)*111.32
    return x,y

def point_polyline_dist_km(px,py,x,y):
    best=np.full(len(px),np.inf)
    for i in range(len(x)-1):
        ax,ay=x[i],y[i]; bx,by=x[i+1],y[i+1]
        vx,vy=bx-ax,by-ay
        den=vx*vx+vy*vy
        if den<1e-12:
            d=np.hypot(px-ax,py-ay)
        else:
            t=np.clip(((px-ax)*vx+(py-ay)*vy)/den,0,1)
            qx=ax+t*vx; qy=ay+t*vy
            d=np.hypot(px-qx,py-qy)
        best=np.minimum(best,d)
    return best

def cross_track_true_to_pred(true_lon,true_lat,pred_lon,pred_lat):
    lon0=np.mean(true_lon); lat0=np.mean(true_lat)
    tx,ty=to_xy_km(true_lon,true_lat,lat0,lon0)
    px,py=to_xy_km(pred_lon,pred_lat,lat0,lon0)
    return point_polyline_dist_km(tx,ty,px,py)

def evaluate_paths(pred1,pred2,true1,true2,issues,model_name,seed):
    rows=[]
    for i,issue in enumerate(issues):
        for j,lead in enumerate(cfg.LEAD_DAYS):
            plon,plat=join_path(pred1[i,j],pred2[i,j])
            tlon,tlat=join_path(true1[i,j],true2[i,j])
            d=cross_track_true_to_pred(tlon,tlat,plon,plat)
            rows.append({
                "model":model_name,"seed":seed,"issue":str(pd.Timestamp(issue).date()),"lead_day":lead,
                "cte_mean_km":float(np.mean(d)),
                "cte_rmse_km":float(np.sqrt(np.mean(d*d))),
                "cte_median_km":float(np.median(d)),
                "cte_p90_km":float(np.percentile(d,90)),
            })
    return pd.DataFrame(rows)


In [ ]:

# ===== 10. Baselines: Persistence / Climatology / AR-EOF =====
from sklearn.linear_model import Ridge

def persistence_pred(data, branch):
    arr=data["test"]
    M = B1.shape[1] if branch==1 else B2.shape[1]
    # X前半是标准化原序列，最后一步反标准化
    pp=pp1 if branch==1 else pp2
    last_std=arr["X"][:,-1,:M]
    last=last_std*pp["sd"]+pp["mu"]
    return np.repeat(last[:,None,:],len(cfg.LEAD_DAYS),axis=1)

def climatology_pred(branch):
    df=df1 if branch==1 else df2
    train=df.loc[:train_end].dropna()
    mean=train.mean(axis=0).to_numpy(np.float32)
    n=len(D1["test"]["issue"])
    return np.repeat(mean[None,None,:],n*len(cfg.LEAD_DAYS),axis=0).reshape(n,len(cfg.LEAD_DAYS),-1)

def ar_eof_fit_predict(data, pp, alpha=1.0):
    # 使用每个输入窗口最后AR_LAGS个PC，直接多输出未来3个lead的PC
    def feats(arr):
        M=len(pp["mu"])
        xs=arr["X"][:,:,:M]  # standardized original only
        pcs=xs @ pp["V"]
        return pcs[:,-cfg.AR_LAGS:,:].reshape(len(xs),-1)
    Xtr=feats(data["train"]); Ytr=data["train"]["Ypc"].reshape(len(Xtr),-1)
    reg=Ridge(alpha=alpha).fit(Xtr,Ytr)
    Xte=feats(data["test"])
    p=reg.predict(Xte).reshape(len(Xte),len(cfg.LEAD_DAYS),pp["V"].shape[1])
    return p.astype(np.float32)

def reconstruct_many(pc,pp):
    shp=pc.shape
    return from_pc(pc.reshape(-1,shp[-1]),pp).reshape(shp[0],shp[1],-1)

base_results=[]
# persistence
p1=persistence_pred(D1,1); p2=persistence_pred(D2,2)
base_results.append(evaluate_paths(p1,p2,D1["test"]["Ypath"],D2["test"]["Ypath"],
                                   D1["test"]["issue"],"Persistence",-1))
# climatology
c1=climatology_pred(1); c2=climatology_pred(2)
base_results.append(evaluate_paths(c1,c2,D1["test"]["Ypath"],D2["test"]["Ypath"],
                                   D1["test"]["issue"],"Climatology",-1))
# AR-EOF
a1=reconstruct_many(ar_eof_fit_predict(D1,pp1),pp1)
a2=reconstruct_many(ar_eof_fit_predict(D2,pp2),pp2)
base_results.append(evaluate_paths(a1,a2,D1["test"]["Ypath"],D2["test"]["Ypath"],
                                   D1["test"]["issue"],"AR-EOF",-1))
baseline_metrics=pd.concat(base_results,ignore_index=True)
display(baseline_metrics.groupby(["model","lead_day"])[["cte_mean_km","cte_rmse_km","cte_p90_km"]].mean())


In [ ]:
# ===== 10.5 FIX2：安全checkpoint + 已完成模型自动续用 =====
SAFE_DIR = OUT / "models_safe_FIX2"
SAFE_DIR.mkdir(parents=True, exist_ok=True)

def safe_ckpt_path(model_name, branch, seed):
    return SAFE_DIR / f"{model_name}_B{branch}_seed{seed}.pt"

def legacy_ckpt_path(model_name, branch, seed):
    return OUT / "models" / f"{model_name}_B{branch}_seed{seed}.pt"

def save_safe_state(model, model_name, branch, seed):
    path = safe_ckpt_path(model_name, branch, seed)
    torch.save({
        "state_dict": {k:v.detach().cpu() for k,v in model.state_dict().items()},
        "model_name": str(model_name),
        "branch": int(branch),
        "seed": int(seed),
    }, path)
    return path

def load_completed_model(model, model_name, branch, seed):
    safe = safe_ckpt_path(model_name, branch, seed)
    if cfg.RESUME_COMPLETED_RUNS and safe.exists():
        obj = torch.load(safe, map_location="cpu", weights_only=True)
        model.load_state_dict(obj["state_dict"])
        print(f"[resume safe] {model_name} B{branch} seed={seed}")
        return model.to(device), True

    legacy = legacy_ckpt_path(model_name, branch, seed)
    if cfg.RESUME_COMPLETED_RUNS and cfg.ALLOW_TRUSTED_LEGACY_RESUME and legacy.exists():
        print(f"[migrate trusted local legacy] {model_name} B{branch} seed={seed}")
        obj = torch.load(legacy, map_location="cpu", weights_only=False)
        model.load_state_dict(obj["state_dict"], strict=True)
        model = model.to(device)
        save_safe_state(model, model_name, branch, seed)
        return model, True

    return model.to(device), False


In [ ]:
# ===== 11. 训练四套原模型 + Simple LSTM；支持断点续跑 =====
specs = [
    ("Simple-LSTM","simple","lstm"),
    ("EOF-Conv-LSTM","conv","lstm"),
    ("EOF-Conv-GRU","conv","gru"),
    ("EOF-TCN-LSTM","tcn","lstm"),
    ("EOF-TCN-GRU","tcn","gru"),
]
all_metrics = [baseline_metrics]
run_rows = []

for name,conv_kind,rnn_kind in specs:
    for seed in cfg.SEEDS:
        print("\n" + "="*90)
        print(name, "seed", seed)

        net1 = build_model(name,conv_kind,rnn_kind,1)
        net1,resumed1 = load_completed_model(net1,name,1,seed)
        if resumed1:
            h1 = pd.DataFrame(); b1 = {"epoch":np.nan,"loss":np.nan}
        else:
            net1,h1,b1 = train_one(net1,D1,cfg.EPOCHS_B1,cfg.LR_DROP_B1,seed)
            save_safe_state(net1,name,1,seed)

        net2 = build_model(name,conv_kind,rnn_kind,2)
        net2,resumed2 = load_completed_model(net2,name,2,seed)
        if resumed2:
            h2 = pd.DataFrame(); b2 = {"epoch":np.nan,"loss":np.nan}
        else:
            net2,h2,b2 = train_one(net2,D2,cfg.EPOCHS_B2,cfg.LR_DROP_B2,seed)
            save_safe_state(net2,name,2,seed)

        pc1 = predict_pc(net1,D1["test"])
        pc2 = predict_pc(net2,D2["test"])
        pred1 = reconstruct_many(pc1,pp1)
        pred2 = reconstruct_many(pc2,pp2)
        met = evaluate_paths(
            pred1,pred2,D1["test"]["Ypath"],D2["test"]["Ypath"],
            D1["test"]["issue"],name,seed
        )
        all_metrics.append(met)

        if len(h1):
            h1.to_csv(OUT/"reports"/f"{name}_B1_seed{seed}_history_FIX2.csv", index=False)
        if len(h2):
            h2.to_csv(OUT/"reports"/f"{name}_B2_seed{seed}_history_FIX2.csv", index=False)

        run_rows.append({
            "model":name,"seed":seed,
            "b1_resumed":resumed1,"b2_resumed":resumed2,
            "b1_best_epoch":b1["epoch"],"b1_val_loss":b1["loss"],
            "b2_best_epoch":b2["epoch"],"b2_val_loss":b2["loss"],
        })

        pd.concat(all_metrics,ignore_index=True).to_csv(
            OUT/"reports"/"all_test_metrics_PARTIAL_FIX2.csv", index=False
        )

metrics = pd.concat(all_metrics,ignore_index=True)
metrics.to_csv(OUT/"reports"/"all_test_metrics_FIX2.csv", index=False)
pd.DataFrame(run_rows).to_csv(OUT/"reports"/"training_runs_FIX2.csv", index=False)
print("saved:", OUT/"reports"/"all_test_metrics_FIX2.csv")


In [ ]:

# ===== 12. 汇总 mean±std，并相对 Persistence 计算 skill =====
summary=(metrics.groupby(["model","lead_day"])
         .agg(cte_mean_km=("cte_mean_km","mean"),
              cte_mean_std=("cte_mean_km","std"),
              cte_rmse_km=("cte_rmse_km","mean"),
              cte_rmse_std=("cte_rmse_km","std"),
              cte_p90_km=("cte_p90_km","mean"),
              n=("cte_mean_km","size"))
         .reset_index())

pers=summary[summary.model=="Persistence"][["lead_day","cte_rmse_km"]].rename(columns={"cte_rmse_km":"pers_rmse"})
summary=summary.merge(pers,on="lead_day",how="left")
summary["skill_vs_persistence"]=1-summary["cte_rmse_km"]/summary["pers_rmse"]
summary.to_csv(OUT/"reports"/"model_summary.csv",index=False)
display(summary.sort_values(["lead_day","cte_rmse_km"]))


In [ ]:

# ===== 13. 资格 Gate =====
# R2 的目标不是宣布最终SOTA，而是判断：日尺度版本是否值得扩展到多年正式hindcast。
gate={}
gate["selected_lookback_days"] = int(cfg.LOOKBACK_DAYS)
gate["lead_days"] = list(cfg.LEAD_DAYS)
gate["train_samples"]=0 if D1["train"] is None else len(D1["train"]["X"])
gate["val_samples"]=0 if D1["val"] is None else len(D1["val"]["X"])
gate["test_samples"]=0 if D1["test"] is None else len(D1["test"]["X"])
gate["b1_eof_cum"]=float(pp1["explained"].sum())
gate["b2_eof_cum"]=float(pp2["explained"].sum())
gate["has_persistence"]=bool((summary.model=="Persistence").any())
gate["has_climatology"]=bool((summary.model=="Climatology").any())
gate["has_ar_eof"]=bool((summary.model=="AR-EOF").any())
gate["has_simple_lstm"]=bool((summary.model=="Simple-LSTM").any())
gate["recommend_multiyear_next"]=True

with open(OUT/"reports"/"R2_HANDOFF.json","w",encoding="utf-8") as f:
    json.dump(gate,f,indent=2,ensure_ascii=False)
print(json.dumps(gate,indent=2,ensure_ascii=False))
print("\nR2通过后：扩展更长日平均年份，做真正 multi-year rolling hindcast。")



## FIX1 运行与诊断说明

如果原 R2 已经报：

```text
KeyError: 'split'
```

请不要只从原来的 split cell 重跑。应使用本 FIX1 并从：

```text
第 2 节：读取R1双分支逐日路径
```

开始顺序执行，因为 anchor 的构建逻辑已经改变。

重点看：

```text
observed path coverage
calendar missing days
longest calendar missing run
Anchor / Lookback 资格审计
selected LOOKBACK_DAYS
split counts
```

若 FIX1 在 `Anchor Gate` 主动停止，请把：

```text
R2_prediction/reports/anchor_lookback_audit.csv
R2_prediction/reports/R2_daily_path_coverage.csv
```

回传。此时应该修 R1 路径覆盖，而不是随机拆分或插值预测真值。
